In [ ]:
# In[1]:

import os
import pandas as pd

# file list and expected name-column mapping
files_info = {
    "metric.csv": "kpi_name",
    "log.csv": "log_name",
    "trace.csv": "trace_name",
    "error_logs.csv": "message",  # error_logs has message column, but user asked for KPI/trace/log names -> will report presence/absence
}

summaries = []

for fname, name_col in files_info.items():
    summary = {"file": fname}
    if not os.path.exists(fname):
        summary.update({
            "row_count": None,
            "cmdb_ids_sample": f"missing file: {fname}",
            "name_column_present": False,
            "name_keys_sample": f"missing file: {fname}",
            "min_timestamp_utc": None,
            "max_timestamp_utc": None
        })
        summaries.append(summary)
        continue

    # load CSV
    try:
        df = pd.read_csv(fname)
    except Exception as e:
        summary.update({
            "row_count": None,
            "cmdb_ids_sample": f"read error: {e}",
            "name_column_present": False,
            "name_keys_sample": f"read error: {e}",
            "min_timestamp_utc": None,
            "max_timestamp_utc": None
        })
        summaries.append(summary)
        continue

    # row count
    summary["row_count"] = int(df.shape[0])

    # cmdb_id unique sample up to 20
    if "cmdb_id" in df.columns:
        cmdb_unique = pd.Series(df["cmdb_id"].dropna().unique())
        cmdb_sample = list(cmdb_unique[:20])
        summary["cmdb_ids_sample"] = cmdb_sample
        summary["cmdb_unique_count"] = int(cmdb_unique.shape[0])
    else:
        summary["cmdb_ids_sample"] = "cmdb_id column missing"
        summary["cmdb_unique_count"] = 0

    # name column presence and sample keys (limit 20)
    if name_col in df.columns:
        name_unique = pd.Series(df[name_col].dropna().unique())
        name_sample = list(name_unique[:20])
        summary["name_column_present"] = True
        summary["name_keys_sample"] = name_sample
        summary["name_keys_count"] = int(name_unique.shape[0])
    else:
        # If the specific expected name column is absent, report that fact
        present_cols = list(df.columns)
        summary["name_column_present"] = False
        summary["name_keys_sample"] = f"expected column '{name_col}' not found; available cols: {present_cols}"

    # timestamps: parse if present
    if "timestamp" in df.columns:
        # parse as UTC datetimes per instructions
        try:
            ts = pd.to_datetime(df["timestamp"], unit="s", utc=True)
            if ts.dropna().empty:
                summary["min_timestamp_utc"] = None
                summary["max_timestamp_utc"] = None
            else:
                summary["min_timestamp_utc"] = ts.min()
                summary["max_timestamp_utc"] = ts.max()
        except Exception as e:
            summary["min_timestamp_utc"] = f"parse error: {e}"
            summary["max_timestamp_utc"] = f"parse error: {e}"
    else:
        summary["min_timestamp_utc"] = "timestamp column missing"
        summary["max_timestamp_utc"] = "timestamp column missing"

    summaries.append(summary)

# create compact DataFrame summary and display it
summary_df = pd.DataFrame(summaries)[[
    "file", "row_count", "cmdb_unique_count", "cmdb_ids_sample",
    "name_column_present", "name_keys_count", "name_keys_sample",
    "min_timestamp_utc", "max_timestamp_utc"
]]

# For compactness, replace NaN with None
summary_df = summary_df.where(pd.notnull(summary_df), None)

summary_df

```
Out[1]:
```
```python
# Summarize the earlier execution results in plain English
summary_text = (
    "metric.csv: 1878 rows, 16 unique cmdb_id values (sample includes carts, carts-db, catalogue, catalogue-db, ...), "
    "7 KPI keys (sample: cpu, diskio, latency-50, latency-90, mem, ...). Time range: 2024-01-20 22:31:00+00:00 to 2024-01-20 22:55:00+00:00 UTC.\n\n"
    "log.csv: 452 rows, 11 unique cmdb_id values (sample includes catalogue, front-end, orders, payment, ...), "
    "2 log keys: log.error_count, log.row_count. Time range: 2024-01-20 22:31:00+00:00 to 2024-01-20 22:55:00+00:00 UTC.\n\n"
    "trace.csv: 0 rows (empty). No trace keys and no timestamps available.\n\n"
    "error_logs.csv: 0 rows (empty). No message entries and no timestamps available."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

file  row_count  cmdb_unique_count                                    cmdb_ids_sample  name_column_present  name_keys_count                                   name_keys_sample         min_timestamp_utc         max_timestamp_utc
0      metric.csv       1878                 16  [carts, carts-db, catalogue, catalogue-db, fro...                 True                7  [cpu, diskio, latency-50, latency-90, mem, soc... 2024-01-20 22:31:00+00:00 2024-01-20 22:55:00+00:00
1         log.csv        452                 11  [catalogue, front-end, orders, payment, queue-...                 True                2                   [log.error_count, log.row_count] 2024-01-20 22:31:00+00:00 2024-01-20 22:55:00+00:00
2       trace.csv          0                  0                                                 []                 True                0                                                 []                       NaT                       NaT
3  error_logs.csv          0                  0                                                 []                 True                0                                                 []                       NaT                       NaT```
```

In [ ]:
# In[2]:

import pandas as pd

# load metric.csv and parse timestamps as UTC datetimes
metric_df = pd.read_csv("metric.csv")
metric_df["timestamp"] = pd.to_datetime(metric_df["timestamp"], unit="s", utc=True)

# 1) Compute global group statistics (using full series BEFORE any window filtering)
group_cols = ["cmdb_id", "kpi_name"]
grp = metric_df.groupby(group_cols)["value"]
stats = grp.agg(
    total_points="count",
    P5=lambda x: x.quantile(0.05),
    P50=lambda x: x.quantile(0.50),
    P90=lambda x: x.quantile(0.90),
    P95=lambda x: x.quantile(0.95),
).reset_index()

# 2) Filter metric rows to the incident window (UTC)
start_ts = pd.to_datetime("2024-01-20 22:28:53", utc=True)
end_ts = pd.to_datetime("2024-01-20 22:58:53", utc=True)
metric_window = metric_df[(metric_df["timestamp"] >= start_ts) & (metric_df["timestamp"] <= end_ts)]

# 3) Merge windowed rows with group's P95 to mark anomalies (value >= P95)
mw = metric_window.merge(stats[["cmdb_id", "kpi_name", "P95"]], on=["cmdb_id", "kpi_name"], how="left")
mw["is_anom"] = mw["value"] >= mw["P95"]

# 4) Aggregate windowed metrics per group
window_points = mw.groupby(group_cols)["value"].count().reset_index().rename(columns={"value": "window_points"})
window_anom_count = mw.groupby(group_cols)["is_anom"].sum().reset_index().rename(columns={"is_anom": "window_anomaly_count"})
window_max = mw.groupby(group_cols)["value"].max().reset_index().rename(columns={"value": "window_max_value"})
# earliest anomaly timestamp per group (NaT if none)
mw_anom = mw[mw["is_anom"]]
earliest_anom = mw_anom.groupby(group_cols)["timestamp"].min().reset_index().rename(columns={"timestamp": "earliest_anomaly_timestamp_utc"})

# 5) Combine stats and window aggregates, ensure groups with no window data are included with zeros/NaT
result = stats.merge(window_points, on=group_cols, how="left")
result = result.merge(window_anom_count, on=group_cols, how="left")
result = result.merge(earliest_anom, on=group_cols, how="left")
result = result.merge(window_max, on=group_cols, how="left")

# fill missing numeric window aggregates with 0
result["window_points"] = result["window_points"].fillna(0).astype(int)
result["window_anomaly_count"] = result["window_anomaly_count"].fillna(0).astype(int)
# window_max_value stays NaN if no window data; earliest_anomaly_timestamp_utc stays NaT if none

# select and order columns as requested
result = result[[
    "cmdb_id",
    "kpi_name",
    "total_points",
    "P5",
    "P50",
    "P90",
    "P95",
    "window_points",
    "window_anomaly_count",
    "earliest_anomaly_timestamp_utc",
    "window_max_value",
]]

# sort by window_anomaly_count desc then window_max_value desc, limit to top 20
result_sorted = result.sort_values(by=["window_anomaly_count", "window_max_value"], ascending=[False, False]).head(20)

result_sorted

```
Out[2]:
```
```python
# Plain-English summary of the metric analysis (global percentiles; window: 2024-01-20 22:28:53 to 2024-01-20 22:58:53 UTC)
summary = (
    "Key findings (percentiles computed on each group's full history; window = 2024-01-20 22:28:53 to 2024-01-20 22:58:53 UTC):\n\n"
    "1) Strong socket anomalies: Several `socket` KPIs show every point in the window >= their global P95 (window_points = 25, window_anomaly_count = 25). "
    "Affected components include rabbitmq, carts-db, orders-db, catalogue-db, queue-master, session-db, and rabbitmq-exporter. "
    "Earliest anomaly timestamps are at 2024-01-20 22:31:00+00:00 for those groups and their window maxima equal the P95 values.\n\n"
    "2) Other socket issues: catalogue (service) and payment also show elevated socket counts in the window but with fewer full-window anomalies (catalogue: 20 hits, payment: 12 hits), earliest anomalies later in the window.\n\n"
    "3) Disk I/O groups appear flagged (many groups show window_anomaly_count = window_points = 24–25) but their P95 = 0 and window_max_value = 0, so these are likely non-actionable (zero baseline) rather than true spikes.\n\n"
    "4) Memory (mem) KPIs: several services (catalogue-db, orders, shipping, queue-master, carts, front-end, rabbitmq, carts-db) show small numbers of window anomalies (window_anomaly_count = 2) with modest increases above their P95; earliest mem anomalies occur between 22:40 and 22:54 UTC and window_max_value gives the observed peak memory values.\n\n"
    "Overall: the most striking signals are the socket-related deviations (many groups with all window samples >= P95). Disk I/O flags are due to zero baselines and are likely not meaningful. Memory shows a few small spikes. Prioritize investigating socket errors/latency in the listed services (rabbitmq, DBs, queue-master, session-db, etc.)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id kpi_name  total_points            P5           P50           P90           P95  window_points  window_anomaly_count earliest_anomaly_timestamp_utc  window_max_value
52           rabbitmq   socket            25  1.100000e+01  1.100000e+01  1.100000e+01  1.100000e+01             25                    25      2024-01-20 22:31:00+00:00      1.100000e+01
10           carts-db   socket            25  6.000000e+00  6.000000e+00  6.000000e+00  6.000000e+00             25                    25      2024-01-20 22:31:00+00:00      6.000000e+00
38          orders-db   socket            25  6.000000e+00  6.000000e+00  6.000000e+00  6.000000e+00             25                    25      2024-01-20 22:31:00+00:00      6.000000e+00
20       catalogue-db   socket            25  4.000000e+00  4.000000e+00  4.000000e+00  4.000000e+00             25                    25      2024-01-20 22:31:00+00:00      4.000000e+00
48       queue-master   socket            25  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00             25                    25      2024-01-20 22:31:00+00:00      3.000000e+00
59         session-db   socket            25  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00             25                    25      2024-01-20 22:31:00+00:00      3.000000e+00
55  rabbitmq-exporter   socket            25  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00             25                    25      2024-01-20 22:31:00+00:00      1.000000e+00
1               carts   diskio            25  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00             25                    25      2024-01-20 22:31:00+00:00      0.000000e+00
18       catalogue-db   diskio            25  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00             25                    25      2024-01-20 22:31:00+00:00      0.000000e+00
67               user   diskio            24  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00             24                    24      2024-01-20 22:32:00+00:00      0.000000e+00
15          catalogue   socket            25  5.440000e+00  7.000000e+00  7.000000e+00  7.000000e+00             25                    20      2024-01-20 22:36:00+00:00      7.000000e+00
43            payment   socket            25  3.000000e+00  3.716667e+00  4.000000e+00  4.000000e+00             25                    12      2024-01-20 22:44:00+00:00      4.000000e+00
19       catalogue-db      mem            25  4.306125e+08  4.306158e+08  4.308366e+08  4.308381e+08             25                     2      2024-01-20 22:50:00+00:00      4.308648e+08
32             orders      mem            25  3.125160e+08  3.221135e+08  3.239311e+08  3.242038e+08             25                     2      2024-01-20 22:49:00+00:00      3.242684e+08
63           shipping      mem            25  2.955128e+08  2.999800e+08  3.020025e+08  3.022056e+08             25                     2      2024-01-20 22:54:00+00:00      3.023185e+08
47       queue-master      mem            25  2.526854e+08  2.562580e+08  2.565207e+08  2.568361e+08             25                     2      2024-01-20 22:54:00+00:00      2.569574e+08
4               carts      mem            25  1.915490e+08  1.996130e+08  2.020497e+08  2.023074e+08             25                     2      2024-01-20 22:53:00+00:00      2.023802e+08
24          front-end      mem            25  9.971778e+07  1.017800e+08  1.038609e+08  1.044842e+08             25                     2      2024-01-20 22:40:00+00:00      1.050719e+08
51           rabbitmq      mem            25  8.996789e+07  9.201862e+07  9.274109e+07  9.279838e+07             25                     2      2024-01-20 22:49:00+00:00      9.281625e+07
9            carts-db      mem            25  6.967786e+07  7.089671e+07  7.353498e+07  7.383831e+07             25                     2      2024-01-20 22:54:00+00:00      7.425418e+07```
```

In [ ]:
# In[3]:

import pandas as pd

# reuse variable names where sensible
log_df = pd.read_csv("log.csv")
log_df["timestamp"] = pd.to_datetime(log_df["timestamp"], unit="s", utc=True)
# ensure value is numeric
log_df["value"] = pd.to_numeric(log_df["value"], errors="coerce")

# 1) Global percentiles per group (computed on full series BEFORE window filtering)
group_cols = ["cmdb_id", "log_name"]
grp = log_df.groupby(group_cols)["value"]
group_stats = grp.agg(
    total_points="count",
    P5=lambda x: x.quantile(0.05),
    P50=lambda x: x.quantile(0.50),
    P90=lambda x: x.quantile(0.90),
    P95=lambda x: x.quantile(0.95),
).reset_index()

# 2) Filter to incident window (UTC)
start_ts = pd.to_datetime("2024-01-20 22:28:53", utc=True)
end_ts = pd.to_datetime("2024-01-20 22:58:53", utc=True)
log_window = log_df[(log_df["timestamp"] >= start_ts) & (log_df["timestamp"] <= end_ts)]

# 3) Mark anomalies in the window by comparing against group's P95 (thresholds computed above)
mw = log_window.merge(group_stats[["cmdb_id", "log_name", "P95"]], on=["cmdb_id", "log_name"], how="left")
mw["is_anom"] = mw["value"] >= mw["P95"]

# 4) Aggregate windowed metrics per group
window_points = mw.groupby(group_cols)["value"].count().reset_index().rename(columns={"value": "window_points"})
window_anom_count = mw.groupby(group_cols)["is_anom"].sum().reset_index().rename(columns={"is_anom": "window_anomaly_count"})
window_max = mw.groupby(group_cols)["value"].max().reset_index().rename(columns={"value": "window_max_value"})
# earliest anomaly timestamp per group (NaT if none)
mw_anom = mw[mw["is_anom"]]
earliest_anom = mw_anom.groupby(group_cols)["timestamp"].min().reset_index().rename(columns={"timestamp": "earliest_anomaly_timestamp_utc"})

# 5) Combine stats and window aggregates, include groups with no window hits
result = group_stats.merge(window_points, on=group_cols, how="left")
result = result.merge(window_anom_count, on=group_cols, how="left")
result = result.merge(earliest_anom, on=group_cols, how="left")
result = result.merge(window_max, on=group_cols, how="left")

# fill missing numeric window aggregates with 0
result["window_points"] = result["window_points"].fillna(0).astype(int)
result["window_anomaly_count"] = result["window_anomaly_count"].fillna(0).astype(int)
# earliest_anomaly_timestamp_utc stays NaT if none; window_max_value stays NaN if none

# select and order columns as requested
result = result[[
    "cmdb_id",
    "log_name",
    "total_points",
    "P5",
    "P50",
    "P90",
    "P95",
    "window_points",
    "window_anomaly_count",
    "earliest_anomaly_timestamp_utc",
    "window_max_value",
]]

# sort by window_anomaly_count desc then window_max_value desc, limit to top 20
result_sorted = result.sort_values(by=["window_anomaly_count", "window_max_value"], ascending=[False, False]).head(20)

# brief list (up to 10) of unique cmdb_id values present in log.csv as a small DataFrame
cmdb_ids_df = pd.DataFrame({"cmdb_id": pd.Series(log_df["cmdb_id"].dropna().unique())})

# display compact outputs: top groups table and up to 10 cmdb_id values
result_sorted, cmdb_ids_df.head(10)

```
Out[3]:
```
```python
summary_text = (
    "Log analysis (percentiles computed on each group's full history; window = 2024-01-20 22:28:53 to 2024-01-20 22:58:53 UTC):\n\n"
    "- Many log.error_count groups (catalogue, front-end, orders, payment, queue-master, shipping, user, carts, session-db, user-db) have P95 = 0 and every point in the window >= P95 (window_points = window_anomaly_count). "
    "However, window_max_value is 0 for these groups, so these are likely non-actionable zero-baseline signals rather than true error spikes.\n\n"
    "- log.row_count shows the most meaningful deviations: front-end, user, queue-master, catalogue, orders, payment, shipping and others have small numbers of window anomalies (window_anomaly_count = 2 for several groups). "
    "Example: front-end row_count has P95 ≈ 2053.2, two points in the window exceed P95, earliest at 2024-01-20 22:32:00+00:00, window_max_value = 2097. Similar 2-hit patterns and timestamps ~22:36:00+00:00 are seen for user, queue-master, catalogue, orders, payment, and shipping.\n\n"
    "- session-db and carts-db have consistent small row_count values (session-db row_count = 5 across points; carts-db row_count = 1 across points) and their window data simply equals their high-percentile baseline.\n\n"
    "Conclusion: prioritize investigating elevated log row rates (log.row_count) in front-end, user, queue-master, catalogue, orders, payment, and shipping. Many log.error_count entries are zero-baseline artifacts and likely not actionable.\n\n"
    "Unique cmdb_id values present in log.csv (up to 10): ['catalogue', 'front-end', 'orders', 'payment', 'queue-master', 'shipping', 'user', 'carts-db', 'session-db', 'carts']"
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id         log_name  total_points      P5     P50     P90     P95  window_points  window_anomaly_count earliest_anomaly_timestamp_utc  window_max_value
4      catalogue  log.error_count            25     0.0     0.0     0.0     0.0             25                    25      2024-01-20 22:31:00+00:00                 0
6      front-end  log.error_count            25     0.0     0.0     0.0     0.0             25                    25      2024-01-20 22:31:00+00:00                 0
8         orders  log.error_count            25     0.0     0.0     0.0     0.0             25                    25      2024-01-20 22:31:00+00:00                 0
10       payment  log.error_count            25     0.0     0.0     0.0     0.0             25                    25      2024-01-20 22:31:00+00:00                 0
12  queue-master  log.error_count            25     0.0     0.0     0.0     0.0             25                    25      2024-01-20 22:31:00+00:00                 0
16      shipping  log.error_count            25     0.0     0.0     0.0     0.0             25                    25      2024-01-20 22:31:00+00:00                 0
18          user  log.error_count            25     0.0     0.0     0.0     0.0             25                    25      2024-01-20 22:31:00+00:00                 0
3       carts-db    log.row_count            24     1.0     1.0     1.0     1.0             24                    24      2024-01-20 22:32:00+00:00                 1
2       carts-db  log.error_count            24     0.0     0.0     0.0     0.0             24                    24      2024-01-20 22:32:00+00:00                 0
0          carts  log.error_count            18     0.0     0.0     0.0     0.0             18                    18      2024-01-20 22:33:00+00:00                 0
15    session-db    log.row_count             5     5.0     5.0     5.0     5.0              5                     5      2024-01-20 22:32:00+00:00                 5
14    session-db  log.error_count             5     0.0     0.0     0.0     0.0              5                     5      2024-01-20 22:32:00+00:00                 0
20       user-db  log.error_count             4     0.0     0.0     0.0     0.0              4                     4      2024-01-20 22:39:00+00:00                 0
7      front-end    log.row_count            25  1729.6  1954.0  2035.2  2053.2             25                     2      2024-01-20 22:32:00+00:00              2097
19          user    log.row_count            25   517.2   594.0   604.2   615.4             25                     2      2024-01-20 22:36:00+00:00               636
13  queue-master    log.row_count            25   473.4   558.0   576.0   590.4             25                     2      2024-01-20 22:36:00+00:00               603
5      catalogue    log.row_count            25   142.2   161.0   167.6   170.4             25                     2      2024-01-20 22:36:00+00:00               173
9         orders    log.row_count            25   105.2   124.0   128.0   131.2             25                     2      2024-01-20 22:36:00+00:00               134
11       payment    log.row_count            25    86.4   101.0   104.0   105.6             25                     2      2024-01-20 22:36:00+00:00               107
17      shipping    log.row_count            25    52.6    62.0    64.0    65.6             25                     2      2024-01-20 22:36:00+00:00                67,         cmdb_id
0     catalogue
1     front-end
2        orders
3       payment
4  queue-master
5      shipping
6          user
7      carts-db
8    session-db
9         carts)```
```

In [ ]:
# In[4]:

import pandas as pd

# reuse or load data
metric_df = pd.read_csv("metric.csv")
metric_df["timestamp"] = pd.to_datetime(metric_df["timestamp"], unit="s", utc=True)
metric_df["value"] = pd.to_numeric(metric_df["value"], errors="coerce")

log_df = pd.read_csv("log.csv")
log_df["timestamp"] = pd.to_datetime(log_df["timestamp"], unit="s", utc=True)
log_df["value"] = pd.to_numeric(log_df["value"], errors="coerce")

# incident window (UTC)
start_ts = pd.to_datetime("2024-01-20 22:28:53", utc=True)
end_ts = pd.to_datetime("2024-01-20 22:58:53", utc=True)

# allowed components and name filters
allowed_cmdb = ["carts", "catalogue", "orders", "payment", "user"]
def name_filter_metric(k): 
    return (k.str.contains("socket", na=False)) | (k == "mem")
def name_filter_log(k):
    return (k.str.contains("socket", na=False)) | (k == "log.row_count") | (k == "mem")

# 1) Compute global P95 and total_points for metrics (group BEFORE filtering)
metric_stats = (
    metric_df
    .groupby(["cmdb_id", "kpi_name"])["value"]
    .agg(total_points="count", P95=lambda x: x.quantile(0.95))
    .reset_index()
)

# 2) Windowed metric aggregates
metric_window = metric_df[(metric_df["timestamp"] >= start_ts) & (metric_df["timestamp"] <= end_ts)]
mw = metric_window.merge(metric_stats, on=["cmdb_id", "kpi_name"], how="left")
mw["is_anom"] = mw["value"] >= mw["P95"]

metric_window_points = mw.groupby(["cmdb_id", "kpi_name"])["value"].count().reset_index().rename(columns={"value":"window_points"})
metric_window_anom = mw.groupby(["cmdb_id", "kpi_name"])["is_anom"].sum().reset_index().rename(columns={"is_anom":"window_anomaly_count"})
metric_window_max = mw.groupby(["cmdb_id", "kpi_name"])["value"].max().reset_index().rename(columns={"value":"window_max_value"})
metric_earliest = mw[mw["is_anom"]].groupby(["cmdb_id", "kpi_name"])["timestamp"].min().reset_index().rename(columns={"timestamp":"earliest_anomaly_timestamp_utc"})

metric_result = metric_stats.merge(metric_window_points, on=["cmdb_id","kpi_name"], how="left")
metric_result = metric_result.merge(metric_window_anom, on=["cmdb_id","kpi_name"], how="left")
metric_result = metric_result.merge(metric_earliest, on=["cmdb_id","kpi_name"], how="left")
metric_result = metric_result.merge(metric_window_max, on=["cmdb_id","kpi_name"], how="left")

metric_result["window_points"] = metric_result["window_points"].fillna(0).astype(int)
metric_result["window_anomaly_count"] = metric_result["window_anomaly_count"].fillna(0).astype(int)

# keep relevant columns and rename for union
metric_result = metric_result.rename(columns={"kpi_name":"name"})
metric_result["data_source"] = "metric"
metric_result = metric_result[[
    "data_source", "cmdb_id", "name", "total_points", "P95",
    "window_points", "window_anomaly_count", "earliest_anomaly_timestamp_utc", "window_max_value"
]]

# 3) Compute global P95 and total_points for logs (group BEFORE filtering)
log_stats = (
    log_df
    .groupby(["cmdb_id", "log_name"])["value"]
    .agg(total_points="count", P95=lambda x: x.quantile(0.95))
    .reset_index()
)

# 4) Windowed log aggregates
log_window = log_df[(log_df["timestamp"] >= start_ts) & (log_df["timestamp"] <= end_ts)]
lw = log_window.merge(log_stats, on=["cmdb_id", "log_name"], how="left")
lw["is_anom"] = lw["value"] >= lw["P95"]

log_window_points = lw.groupby(["cmdb_id", "log_name"])["value"].count().reset_index().rename(columns={"value":"window_points"})
log_window_anom = lw.groupby(["cmdb_id", "log_name"])["is_anom"].sum().reset_index().rename(columns={"is_anom":"window_anomaly_count"})
log_window_max = lw.groupby(["cmdb_id", "log_name"])["value"].max().reset_index().rename(columns={"value":"window_max_value"})
log_earliest = lw[lw["is_anom"]].groupby(["cmdb_id", "log_name"])["timestamp"].min().reset_index().rename(columns={"timestamp":"earliest_anomaly_timestamp_utc"})

log_result = log_stats.merge(log_window_points, on=["cmdb_id","log_name"], how="left")
log_result = log_result.merge(log_window_anom, on=["cmdb_id","log_name"], how="left")
log_result = log_result.merge(log_earliest, on=["cmdb_id","log_name"], how="left")
log_result = log_result.merge(log_window_max, on=["cmdb_id","log_name"], how="left")

log_result["window_points"] = log_result["window_points"].fillna(0).astype(int)
log_result["window_anomaly_count"] = log_result["window_anomaly_count"].fillna(0).astype(int)

# rename for union
log_result = log_result.rename(columns={"log_name":"name"})
log_result["data_source"] = "log"
log_result = log_result[[
    "data_source", "cmdb_id", "name", "total_points", "P95",
    "window_points", "window_anomaly_count", "earliest_anomaly_timestamp_utc", "window_max_value"
]]

# 5) Combine metric and log results, filter by allowed components and name conditions
combined = pd.concat([metric_result, log_result], ignore_index=True)

# apply cmdb and name filters
combined = combined[combined["cmdb_id"].isin(allowed_cmdb)]

# name filter: contains 'socket' OR equals 'log.row_count' OR equals 'mem'
name_series = combined["name"].astype(str)
name_mask = name_series.str.contains("socket", na=False) | (name_series == "log.row_count") | (name_series == "mem")
combined = combined[name_mask]

# fill NaT and NaN consistently (earliest stays NaT if none)
combined["window_max_value"] = combined["window_max_value"].astype(float)

# 6) Sort and limit to top 20
final_result = combined.sort_values(by=["window_anomaly_count", "window_max_value"], ascending=[False, False]).head(20)

# keep output compact: selected columns only
final_result = final_result[[
    "data_source", "cmdb_id", "name", "total_points", "P95",
    "window_points", "window_anomaly_count", "earliest_anomaly_timestamp_utc", "window_max_value"
]]

final_result

```
Out[4]:
```
Summary (incident window 2024-01-20 22:28:53–22:58:53 UTC):

- Strongest signal: catalogue.socket — 20 of 25 samples in the window exceeded its global P95 (P95 = 7). Earliest anomaly at 2024-01-20 22:36:00 UTC. This is the highest-priority finding.
- Next notable socket signal: payment.socket — 12 of 25 samples ≥ P95 (P95 = 4), earliest at 2024-01-20 22:44:00 UTC.
- Additional socket activity with fewer hits: user.socket, orders.socket, and carts.socket each show 2 window samples ≥ P95 (small number of spikes; earliest timestamps between ~22:33–22:54 UTC).
- log.row_count increases: user, catalogue, orders, and payment each had 2 window points above their P95 (earliest at ~22:36:00 UTC); carts.log.row_count had 1 hit (earliest ~22:39:00 UTC). These indicate transient elevated request/log rates across front-end/back-end services around 22:36 UTC.
- Memory (mem) shows small, 2-sample spikes for orders, carts, user, catalogue, and payment (modest increases above their P95; earliest between ~22:38–22:53 UTC).
- Overall recommendation: prioritize investigating socket-related issues in catalogue (highest priority) and payment, then examine the brief elevated request/row rates (log.row_count) across user, catalogue, orders and payment. Memory spikes are present but smaller and secondary.

The original code execution output of IPython Kernel is also provided below for reference:

data_source    cmdb_id           name  total_points           P95  window_points  window_anomaly_count earliest_anomaly_timestamp_utc  window_max_value
15      metric  catalogue         socket            25  7.000000e+00             25                    20      2024-01-20 22:36:00+00:00      7.000000e+00
43      metric    payment         socket            25  4.000000e+00             25                    12      2024-01-20 22:44:00+00:00      4.000000e+00
32      metric     orders            mem            25  3.242038e+08             25                     2      2024-01-20 22:49:00+00:00      3.242684e+08
4       metric      carts            mem            25  2.023074e+08             25                     2      2024-01-20 22:53:00+00:00      2.023802e+08
70      metric       user            mem            25  1.634696e+07             25                     2      2024-01-20 22:53:00+00:00      1.722150e+07
14      metric  catalogue            mem            25  5.937744e+06             25                     2      2024-01-20 22:46:00+00:00      5.945617e+06
42      metric    payment            mem            25  5.219970e+06             25                     2      2024-01-20 22:38:00+00:00      5.281997e+06
96         log       user  log.row_count            25  6.154000e+02             25                     2      2024-01-20 22:36:00+00:00      6.360000e+02
82         log  catalogue  log.row_count            25  1.704000e+02             25                     2      2024-01-20 22:36:00+00:00      1.730000e+02
86         log     orders  log.row_count            25  1.312000e+02             25                     2      2024-01-20 22:36:00+00:00      1.340000e+02
88         log    payment  log.row_count            25  1.056000e+02             25                     2      2024-01-20 22:36:00+00:00      1.070000e+02
71      metric       user         socket            25  3.679778e+01             25                     2      2024-01-20 22:54:00+00:00      3.710000e+01
33      metric     orders         socket            25  1.818000e+01             25                     2      2024-01-20 22:45:00+00:00      1.858333e+01
5       metric      carts         socket            25  1.097143e+01             25                     2      2024-01-20 22:33:00+00:00      1.100000e+01
78         log      carts  log.row_count            18  1.037500e+02             18                     1      2024-01-20 22:39:00+00:00      1.250000e+02```
```